# Qwen3-TTS — Clonacion de Voz & Text-to-Speech

3 modos:
- **Voice Clone** — clona cualquier voz con un audio de referencia
- **Voice Design** — describe la voz que quieres en texto natural
- **Custom Voice** — speakers predefinidos con instrucciones de estilo

**Requisitos:** GPU T4 — `Runtime > Change runtime type > T4 GPU`

---

## 1. Instalacion (~3-5 min)

In [ ]:
!pip install -q torch torchaudio
!pip install -q transformers accelerate soundfile numpy
!pip install -q huggingface_hub gradio

# Instalar qwen_tts (el paquete oficial)
!pip install -q qwen-tts

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("NO GPU — activa T4 en Runtime > Change runtime type!")

## 2. Cargar modelo (~3-5 min primera vez)

Usa el modelo **Base 0.6B** para clonacion de voz (cabe bien en T4 16GB).

In [ ]:
from huggingface_hub import snapshot_download
from qwen_tts import Qwen3TTSModel
import torch

# Descargar modelo Base 0.6B (voice cloning)
print("Descargando modelo Base 0.6B...")
base_path = snapshot_download("Qwen/Qwen3-TTS-12Hz-0.6B-Base")

print("Cargando modelo en GPU...")
model = Qwen3TTSModel.from_pretrained(
    base_path,
    device_map="cuda",
    dtype=torch.bfloat16,
)

print("Modelo Base 0.6B cargado!")

## 3. Clonar una voz

Sube un audio de referencia (5-15 segundos) y escribe el texto que quieres que diga con esa voz.

In [ ]:
from google.colab import files
import numpy as np
import soundfile as sf

print("Sube un audio de referencia (WAV, MP3, FLAC — 5-15 segundos):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Audio subido: {ref_audio_path}")

In [ ]:
import torchaudio
from IPython.display import Audio, display
import numpy as np

# Cargar y preparar audio de referencia
ref_waveform, ref_sr = torchaudio.load(ref_audio_path)

# Resamplear a formato compatible
if ref_sr != 24000:
    ref_waveform = torchaudio.functional.resample(ref_waveform, ref_sr, 24000)
    ref_sr = 24000

# Mono
if ref_waveform.shape[0] > 1:
    ref_waveform = ref_waveform.mean(dim=0, keepdim=True)

# Normalizar a float32 [-1, 1]
ref_np = ref_waveform.squeeze().numpy().astype(np.float32)
m = np.max(np.abs(ref_np))
if m > 1.0:
    ref_np = ref_np / m

ref_audio_tuple = (ref_np, ref_sr)

print(f"Audio de referencia: {len(ref_np)/ref_sr:.1f}s, {ref_sr}Hz")
display(Audio(ref_np, rate=ref_sr))

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

# === CONFIGURACION ===
texto = "Hola, esto es una prueba de clonacion de voz con Qwen TTS."
ref_text = ""  # Transcripcion del audio de referencia (dejar vacio para x-vector only)
idioma = "Auto"  # Auto, Chinese, English, Spanish, French, German, Japanese, Korean, Portuguese, Russian
# ====================

use_xvector = (ref_text.strip() == "")

print(f"Modo: {'x-vector only' if use_xvector else 'con transcripcion'}")
print(f"Generando audio...")

wavs, sr = model.generate_voice_clone(
    text=texto,
    language=idioma,
    ref_audio=ref_audio_tuple,
    ref_text=ref_text.strip() if ref_text.strip() else None,
    x_vector_only_mode=use_xvector,
    max_new_tokens=2048,
)

sf.write("output_clonado.wav", wavs[0], sr)
print(f"Audio generado: {len(wavs[0])/sr:.1f}s")
display(Audio(wavs[0], rate=sr))

## 4. Descargar audio

In [ ]:
from google.colab import files
files.download("output_clonado.wav")

---

## 5. Interfaz Gradio (opcional)

Lanza una interfaz web para probar mas comodo. Te da un enlace publico que puedes abrir en el movil.

In [ ]:
import gradio as gr
import numpy as np
import soundfile as sf
import tempfile

LANGUAGES = ["Auto", "Chinese", "English", "Spanish", "French", "German",
             "Japanese", "Korean", "Portuguese", "Russian"]

def normalize_audio(wav):
    x = np.asarray(wav, dtype=np.float32)
    if np.issubdtype(x.dtype, np.integer):
        info = np.iinfo(x.dtype)
        x = x.astype(np.float32) / max(abs(info.min), info.max)
    m = np.max(np.abs(x))
    if m > 1.0:
        x = x / m
    if x.ndim > 1:
        x = np.mean(x, axis=-1).astype(np.float32)
    return np.clip(x, -1.0, 1.0)

def clone_voice(ref_audio, ref_text, target_text, language, use_xvector):
    if not target_text or not target_text.strip():
        return None, "Error: escribe un texto"
    if ref_audio is None:
        return None, "Error: sube un audio de referencia"

    try:
        sr_in, wav_in = ref_audio
        wav_np = normalize_audio(wav_in)

        # Resamplear si hace falta
        if sr_in != 24000:
            import torchaudio
            t = torch.tensor(wav_np).unsqueeze(0)
            t = torchaudio.functional.resample(t, sr_in, 24000)
            wav_np = t.squeeze().numpy()
            sr_in = 24000

        audio_tuple = (wav_np, 24000)

        wavs, sr = model.generate_voice_clone(
            text=target_text.strip(),
            language=language,
            ref_audio=audio_tuple,
            ref_text=ref_text.strip() if ref_text and ref_text.strip() and not use_xvector else None,
            x_vector_only_mode=use_xvector or not (ref_text and ref_text.strip()),
            max_new_tokens=2048,
        )

        return (sr, wavs[0]), f"Generado: {len(wavs[0])/sr:.1f}s"
    except Exception as e:
        return None, f"Error: {e}"

with gr.Blocks(title="Qwen3-TTS — DRIFTLYA", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Qwen3-TTS — Voice Cloning\n### driftlya")

    with gr.Row():
        with gr.Column():
            ref_audio_input = gr.Audio(label="Audio de referencia (5-15s)", type="numpy")
            ref_text_input = gr.Textbox(label="Transcripcion del audio (opcional)", lines=2,
                                        placeholder="Deja vacio para modo x-vector only")
            xvector_check = gr.Checkbox(label="Solo x-vector (sin transcripcion)", value=True)
            target_text_input = gr.Textbox(label="Texto a generar", lines=3,
                                           placeholder="Escribe lo que quieres que diga...")
            lang_dropdown = gr.Dropdown(label="Idioma", choices=LANGUAGES, value="Auto")
            gen_btn = gr.Button("Generar", variant="primary")

        with gr.Column():
            audio_output = gr.Audio(label="Audio generado", type="numpy")
            status_text = gr.Textbox(label="Estado", interactive=False)

    gen_btn.click(
        fn=clone_voice,
        inputs=[ref_audio_input, ref_text_input, target_text_input, lang_dropdown, xvector_check],
        outputs=[audio_output, status_text]
    )

demo.launch(share=True)
print("Copia el enlace 'Running on public URL' para acceder desde el movil")